In [1]:
# dataset.py = loadimage+labels and return tensor version of image
# for converting image into input tensor
from torch.utils.data import Dataset
import os
from PIL import Image
import torch

class SAR_Dataset(Dataset):
    def __init__(self, base_dir, transform=None):
        self.base_dir = base_dir
        self.transform = transform
        self.data = self._build_data()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]        # unpack tuple
        img = Image.open(img_path)
        label = torch.tensor(label, dtype=torch.long)
        if self.transform:
            img = self.transform(img)
        return img, label

    def _build_data(self):
        classes_to_idx = {"agri": 0, "barrenland": 1, "grassland": 2, "urban": 3}
        data = []
        for cls in classes_to_idx:
            s1_folder = os.path.join(self.base_dir, cls, "s1")
            for img in os.listdir(s1_folder):
                img_path = os.path.join(s1_folder, img)
                data.append((img_path, classes_to_idx[cls]))
        return data

In [2]:
model=SAR_Dataset("/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2")
len(model)

16000

In [3]:
# test dataset.py
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Grayscale(),         # ensure 1 channel
    transforms.Resize((64, 64)),    # fix size
    transforms.ToTensor(),          # PIL → tensor (1, 64, 64)
])

dataset = SAR_Dataset("/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2", transform=transform)

# Test 1 - how many images total
print("Total images:", len(dataset))

# Test 2 - look at one sample
img, label = dataset[0]
print("Image tensor shape:", img.shape)   # should be (1, 64, 64)
print("Label:", label)                    # should be tensor(0,1,2 or 3)
print("Min val:", img.min())              # after ToTensor → 0.0 to 1.0
print("Max val:", img.max())

img = Image.open("/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2/agri/s1/ROIs1868_summer_s1_59_p100.png")
print(img.size)   # gives (width, height)

Total images: 16000
Image tensor shape: torch.Size([1, 64, 64])
Label: tensor(0)
Min val: tensor(0.0627)
Max val: tensor(0.9843)
(256, 256)


In [4]:
# model.py  model architechture

import torch.nn as nn

class SAR_model(nn.Module):

    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(   # group 1 , size(256,256)
            nn.Conv2d(1, 32 , 3 , padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 256/2 = size(128,128)
            
            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 128/2 =size(64,64)
            
            nn.Conv2d(64,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 64/2 =size(32,32)
            
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 32/2 = size(16,16)
            
            nn.Conv2d(128,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2) # 16/2 =size(8,8)
            
        )                                     
        self.classifier= nn.Sequential(    # group 2
            nn.Linear(128*8*8 ,128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128,4)
        )

    def forward(self,x):
        x=self.features(x)    # ( batch , hight, width, out_channel )
        x=x.view(x.size(0),-1) #(batch,features(hight*widht*out_channel))
        x=self.classifier(x) # ( batch , 4( final output))
        return x




In [5]:
# utils.py - config+transform+helper

import torch
from torchvision import transforms
from torch.utils.data import random_split,DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES=4
BATCH_SIZE=32
BASE_DIR="/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2"
EPOCHS=20
LR=1e-4


def get_sar_train_transforms():

    return transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((256,256)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5],
            std=[0.5]) ])

def get_sar_val_transforms():
    
    return transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((256,256)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5],
            std=[0.5]) ])

def get_sar_dataloader(base_dir):
    # Two instances, same data, different transforms
    train_dataset = SAR_Dataset(base_dir, transform=get_sar_train_transforms())
    val_dataset   = SAR_Dataset(base_dir, transform=get_sar_val_transforms())

    # Need same indices for both — so split on indices, not dataset objects
    total = len(train_dataset)
    train_size = int(0.8 * total)
    val_size   = total - train_size

    shuffled_index = torch.randperm(total)          # shuffled index list
    train_index = shuffled_index[:train_size]
    val_index   = shuffled_index[train_size:]

    # Subset with random index
    train_subset = torch.utils.data.Subset(train_dataset, train_index)
    val_subset   = torch.utils.data.Subset(val_dataset,   val_index)

    # Wrap in DataLoaders
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader


    
    


In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [7]:
# train.py - training + evalution
from tqdm import tqdm
def train_one_epoch(model,criterion,optimizer,loader,device):
    model.train()
    total_loss=0

    loop = tqdm(loader, desc="Training", leave=False)
    for images,labels in loop:
        images=images.to(device)
        labels=labels.to(device)

        optimizer.zero_grad()
        output=model(images)
        loss=criterion(output,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()

        loop.set_postfix(loss=loss.item())
        
    return total_loss/len(loader)

def evaluate(model,loader,device):
    model.eval()
    correct=0
    total=0

    with torch.no_grad():
        loop = tqdm(loader, desc="evaluating", leave=False)
        for images,labels in loop:
            images=images.to(device)
            labels=labels.to(device)

            output=model(images)
            preds = torch.argmax(output, dim=1)  # index of highest logit = predicted class
            correct+=(preds==labels).sum().item()
            total+=labels.numel()

            loop.set_postfix(acc=f"{correct/total*100:.2f}%")

    return correct/total



In [8]:
# main.py join every thing
import torch.nn as nn
import torch.optim as optim

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def main():
    MODEL = SAR_model().to(DEVICE)
    CRITERION = nn.CrossEntropyLoss()
    OPTIMIZER = optim.Adam(MODEL.parameters(), lr=LR)
    best_val_acc = 0.0
    train_loader, val_loader = get_sar_dataloader(BASE_DIR)

    # load checkpoint if exists
    checkpoint_path = "best_model.pth"
    start_epoch = 0

    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        MODEL.load_state_dict(checkpoint["model_state"])
        OPTIMIZER.load_state_dict(checkpoint["optimizer_state"])
        best_val_acc = checkpoint["best_val_acc"]
        start_epoch = checkpoint["epoch"] + 1
        print(f"Resuming from epoch {start_epoch} | Best val acc so far: {best_val_acc*100:.2f}%")

    for epoch in range(start_epoch, EPOCHS):
        train_loss = train_one_epoch(model=MODEL, criterion=CRITERION,
                                      optimizer=OPTIMIZER, loader=train_loader, device=DEVICE)
        val_acc = evaluate(model=MODEL, loader=val_loader, device=DEVICE)

        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "epoch"          : epoch,
                "model_state"    : MODEL.state_dict(),
                "optimizer_state": OPTIMIZER.state_dict(),
                "best_val_acc"   : best_val_acc
            }, checkpoint_path)
            print(f"  → checkpoint saved (val acc: {best_val_acc*100:.2f}%)")

if __name__ == "__main__":
    main()

Epoch 1/20 | Train Loss: 0.5928 | Val Acc: 87.75%
  → checkpoint saved (val acc: 87.75%)


Epoch 2/20 | Train Loss: 0.2883 | Val Acc: 82.28%


Epoch 3/20 | Train Loss: 0.2287 | Val Acc: 88.06%
  → checkpoint saved (val acc: 88.06%)


Epoch 4/20 | Train Loss: 0.1749 | Val Acc: 91.03%
  → checkpoint saved (val acc: 91.03%)


Epoch 5/20 | Train Loss: 0.1541 | Val Acc: 81.78%


...

Epoch 16/20 | Train Loss: 0.0607 | Val Acc: 98.22%


Epoch 17/20 | Train Loss: 0.0574 | Val Acc: 96.56%


Epoch 18/20 | Train Loss: 0.0529 | Val Acc: 94.72%


Epoch 19/20 | Train Loss: 0.0600 | Val Acc: 98.66%
  → checkpoint saved (val acc: 98.66%)


Epoch 20/20 | Train Loss: 0.0427 | Val Acc: 99.09%
  → checkpoint saved (val acc: 99.09%)


In [9]:
# model.py  model architechture

import torch.nn as nn

class OPTICAL_model(nn.Module):
# 
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(   # group 1 , size(256,256)
            nn.Conv2d(3, 32 , 3 , padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 256/2 = size(128,128)
            
            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 128/2 =size(64,64)
            
            nn.Conv2d(64,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 64/2 =size(32,32)
            
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # 32/2 = size(16,16)
            
            nn.Conv2d(128,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2) # 16/2 =size(8,8)
            
        )                                     
        self.classifier= nn.Sequential(    # group 2
            nn.Linear(128*8*8 ,128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128,4)
        )

    def forward(self,x):
        x=self.features(x)    # ( batch , hight, width, out_channel )
        x=x.view(x.size(0),-1) #(batch,features(hight*widht*out_channel))
        x=self.classifier(x) # ( batch , 4( final output))
        return x




In [10]:
# utils.py - config+transform+helper

import torch
from torchvision import transforms
from torch.utils.data import random_split,DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES=4
BATCH_SIZE=32
BASE_DIR="/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2"
EPOCHS=20
LR=1e-4


def get_sar_train_transforms():

    return transforms.Compose([
        transforms.Resize((256,256)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]) ])

def get_sar_val_transforms():
    
    return transforms.Compose([
        transforms.Resize((256,256)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]) ])

def get_sar_dataloader(base_dir):
    # Two instances, same data, different transforms
    train_dataset = OPTICAL_Dataset(base_dir, transform=get_sar_train_transforms())
    val_dataset   = OPTICAL_Dataset(base_dir, transform=get_sar_val_transforms())

    # Need same indices for both — so split on indices, not dataset objects
    total = len(train_dataset)
    train_size = int(0.8 * total)
    val_size   = total - train_size

    shuffled_index = torch.randperm(total)          # shuffled index list
    train_index = shuffled_index[:train_size]
    val_index   = shuffled_index[train_size:]

    # Subset with random index
    train_subset = torch.utils.data.Subset(train_dataset, train_index)
    val_subset   = torch.utils.data.Subset(val_dataset,   val_index)

    # Wrap in DataLoaders
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader


    
    


In [11]:
# train.py - training + evalution
from tqdm import tqdm
import torch
def train_one_epoch(model,criterion,optimizer,loader,device):
    model.train()
    total_loss=0

    loop = tqdm(loader, desc="Training", leave=False)
    for images,labels in loop:
        images=images.to(device)
        labels=labels.to(device)

        optimizer.zero_grad()
        output=model(images)
        loss=criterion(output,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()

        loop.set_postfix(loss=loss.item())
        
    return total_loss/len(loader)

def evaluate(model,loader,device):
    model.eval()
    correct=0
    total=0

    with torch.no_grad():
        loop = tqdm(loader, desc="evaluating", leave=False)
        for images,labels in loop:
            images=images.to(device)
            labels=labels.to(device)

            output=model(images)
            preds = torch.argmax(output, dim=1)  # index of highest logit = predicted class
            correct+=(preds==labels).sum().item()
            total+=labels.numel()

            # train.py - training + evalution
from tqdm import tqdm
def train_one_epoch(model,criterion,optimizer,loader,device):
    model.train()
    total_loss=0

    loop = tqdm(loader, desc="Training", leave=False)
    for images,labels in loop:
        images=images.to(device)
        labels=labels.to(device)

        optimizer.zero_grad()
        output=model(images)
        loss=criterion(output,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()

        loop.set_postfix(loss=loss.item())
        
    return total_loss/len(loader)

def evaluate(model,loader,device):
    model.eval()
    correct=0
    total=0

    with torch.no_grad():
        loop = tqdm(loader, desc="evaluating", leave=False)
        for images,labels in loop:
            images=images.to(device)
            labels=labels.to(device)

            output=model(images)
            preds = torch.argmax(output, dim=1)  # index of highest logit = predicted class
            correct+=(preds==labels).sum().item()
            total+=labels.numel()

            loop.set_postfix(acc=f"{correct/total*100:.2f}%")

    return correct/total

